# Exercises XP Gold: Titanic. Student Notebook

This notebook is guided to complete the execices on the plateforme. 
For each exercise, the **Instructions** are guided and some parts are already coded, and the **Guidance** explains exactly what you must do in complete sentences.

All **Learning Point** sections thoughout this notebook will provide you important concepts and insights to retain.

---

In these exercises, I predict which Titanic passengers survived. I clean the data, explore it, and train a Decision Tree, a KNN and a Neural Network, each one first with settings I choose myself and then with tuning. At the end, I compare all the models. I follow the course's pre-filled Colab notebook and complete its TODO parts.

## What will you create
- A comprehensive machine learning project focused on predicting the survival of passengers on the Titanic.
- A series of robust classification models trained on the Titanic dataset.
- Detailed comparison reports of model performances with and without hyperparameter tuning.
- Customized hyperparameter tuning processes for different types of models.
- Visualizations of data insights, model performances, and feature importances.
- A final, well-documented Jupyter Notebook showcasing the entire analysis process.

## What will you learn
- Skills in data preparation, including dealing with missing values and feature engineering specifically for the Titanic dataset.
- Understanding of model-specific data preprocessing requirements.
- Experience in building, evaluating, and tuning various classification models like Decision Trees, K-Nearest Neighbors (KNN), and Neural Networks.
- Advanced techniques in hyperparameter tuning using Grid Search and Randomized Search.
- Deep insights into model evaluation metrics specific to classification tasks like accuracy, precision, recall, and F1 score.
- Development of a systematic approach to compare different machine learning models and select the best one for a given problem.
- For the below exercises, you will use the titanic dataset

## Dataset
If a Titanic CSV file is available in your working directory or `./data` (for example, `titanic.csv` with a `Survived` column), you will load it. 
If it is not available, a synthetic Titanic-like dataset will be generated so you can complete the exercises.

## How to run in Colab
1. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
2. I keep the course zip from the [course link](https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%204/Day%203/Titanic%20Dataset.zip) in my `C:\Users\sivan\Datasets` folder as `Titanic-Dataset.zip`. I saved it with a dash, because my folder already has a different `titanic dataset.zip` (the Kaggle train/test files), and Windows sees the two names as the same.
3. I run the cells from top to bottom. When the upload cell opens a file picker, I choose `Titanic-Dataset.zip`.

In [ ]:
from google.colab import files
# I import Colab's tool for uploading files from my computer.
uploaded = files.upload()
# I open a file picker and choose "Titanic-Dataset.zip".
!unzip -o "Titanic-Dataset.zip"
# I extract Titanic-Dataset.csv from the zip into Colab.

### Learning point
You improve faster by making small, clear steps. Load the data, clean it, split it, and only then try models. Keep the target separate from features to avoid leaking answers.

## Exercise 1: Exploratory Data Analysis

### Instructions
Load the Titanic dataset.
Perform data cleaning and handle missing values.
Conduct exploratory data analysis to understand the relationships between different features and survival.

In this exercise, I load the real Titanic CSV, fill or drop the missing values, add a `FamilySize` feature, and look at how sex, class, age, port and family size relate to survival. Then I split the data into a training set and a testing set.

### What I learn here
- I find the missing values and choose how to handle each column.
- I create a new feature from two existing ones (feature engineering).
- I use simple counts and plots to see which passengers survived more.

### Learning point
EDA is about asking simple questions first. How many people survived? Does class or gender seem related to survival? Simple counts and plots tell useful stories.

In [ ]:
# I load the Titanic dataset into a DataFrame named df.
# The target column is 'Survived'. It is 0 when the passenger died and 1 when the passenger survived.
# The course file already uses the standard Titanic column names, so I do not need to rename anything.
# I print df.shape, df.dtypes.head(), the total missing values, and the counts of 'Survived'.

import pandas as pd
from pathlib import Path

csv_path = Path('Titanic-Dataset.csv')  # the file I extracted from the zip
df = pd.read_csv(csv_path)
# I make sure the target column is there before I continue.
assert 'Survived' in df.columns, "Expected a 'Survived' target column"

print('Shape:', df.shape)
print(df.dtypes.head())
print('Total missing values:', df.isna().sum().sum())
print('Missing values per column:')
print(df.isna().sum()[df.isna().sum() > 0])
print('Survived counts:', df['Survived'].value_counts().to_dict())
display(df.head())

In [ ]:
# This piece of code is already prefilled, run it to execute it and see the results.
# It creates a synthetic Titanic-like dataset if you did not load 'df' above.

import pandas as pd
import numpy as np

try:
    df  # will raise NameError if not defined
    print("Dataset already loaded.")
except NameError:
    rng = np.random.default_rng(42)
    n = 1000
    df = pd.DataFrame({
        "Pclass": rng.integers(1, 4, size=n),
        "Sex": rng.choice(["male","female"], size=n),
        "Age": np.clip(rng.normal(30, 14, size=n), 0, None),
        "SibSp": rng.integers(0, 5, size=n),
        "Parch": rng.integers(0, 4, size=n),
        "Fare": np.abs(rng.normal(32, 50, size=n)),
        "Embarked": rng.choice(["S","C","Q"], size=n, p=[0.7,0.2,0.1]),
    })
    # Simple survival rule + noise to mimic correlations
    base = 0.3 + 0.15*(df["Sex"]=="female") + 0.1*(df["Pclass"]==1) - 0.1*(df["Pclass"]==3) + 0.05*(df["Fare"]>40)
    df["Survived"] = (rng.random(n) < base.clip(0,1)).astype(int)
    # Add some missingness
    miss_idx = rng.choice(n, size=int(0.1*n), replace=False)
    df.loc[miss_idx, "Age"] = np.nan
    print("Synthetic Titanic-like dataset created. Shape:", df.shape)
    print(df.head(3))

**What I see:** The real dataset has **891 passengers** and 12 columns, so the template's synthetic data is not used (the cell prints "Dataset already loaded."). There are **866 missing values**: 177 in `Age`, 687 in `Cabin` and 2 in `Embarked`. **549 passengers died** (0) and **342 survived** (1), so about 38% survived.

In [ ]:
# I create a cleaned DataFrame named df_clean.
# I fill the missing numeric values (like Age) with the median and the missing categorical values (like Embarked) with the most frequent value.
# I also create a simple 'FamilySize' feature as SibSp + Parch + 1 and keep it in df_clean.
# I print the number of remaining missing values in df_clean.

import pandas as pd

# I drop four columns: Cabin is missing for most passengers, and PassengerId, Name and Ticket
# are almost unique for every passenger, so they cannot help the model find patterns.
df_clean = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], errors='ignore').copy()

# The median is the middle value, so a few very old or very young passengers do not pull it.
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
# mode()[0] is the most frequent port.
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

# FamilySize = siblings/spouses + parents/children + the passenger.
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1

print('Median age used:', df['Age'].median(), '| Most frequent port used:', df['Embarked'].mode()[0])
print('Remaining missing values in df_clean:', df_clean.isna().sum().sum())
print('Columns:', df_clean.columns.tolist())

In [ ]:
# This piece of code is already prefilled, run it to execute it and see the results.
# It shows simple plots for survival counts, survival by Pclass, and an Age histogram.

import matplotlib.pyplot as plt

surv_counts = df_clean["Survived"].value_counts().sort_index()
plt.figure()
surv_counts.plot(kind="bar")
plt.title("Survival counts (0=No, 1=Yes)")
plt.xlabel("Survived")
plt.ylabel("Count")
plt.show()

pclass_surv = df_clean.groupby("Pclass")["Survived"].mean()
plt.figure()
pclass_surv.plot(kind="bar")
plt.title("Mean survival rate by Pclass")
plt.xlabel("Pclass")
plt.ylabel("Mean survival")
plt.show()

plt.figure()
df_clean["Age"].hist(bins=20)
plt.title("Age distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

In this cell, I look at more relationships between the features and survival: the survival rate by sex, by port of embarkation and by family size, and the ages of passengers who survived and who did not.

In [ ]:
# The mean of a 0/1 column is the share of 1s, so this is the survival rate of each group.
print('Survival rate by Sex:')
print(df_clean.groupby('Sex')['Survived'].mean().round(3))
print('\nSurvival rate by Embarked (C = Cherbourg, Q = Queenstown, S = Southampton):')
print(df_clean.groupby('Embarked')['Survived'].mean().round(3))
# The survival rate of each class, split by sex.
print('\nSurvival rate by Pclass and Sex:')
print(df_clean.pivot_table(index='Pclass', columns='Sex', values='Survived', aggfunc='mean').round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df_clean.groupby('Sex')['Survived'].mean().plot(kind='bar', ax=axes[0], title='Survival rate by Sex')
df_clean.groupby('FamilySize')['Survived'].mean().plot(kind='bar', ax=axes[1], title='Survival rate by FamilySize')
# I draw the age histogram twice: once for the passengers who died and once for the survivors.
df_clean[df_clean['Survived'] == 0]['Age'].hist(bins=20, alpha=0.6, label='Died (0)', ax=axes[2])
df_clean[df_clean['Survived'] == 1]['Age'].hist(bins=20, alpha=0.6, label='Survived (1)', ax=axes[2])
axes[2].set_title('Age by survival')
axes[2].legend()
plt.tight_layout()
plt.show()

# Correlation of every number column with Survived (from -1 to 1).
print(df_clean.select_dtypes(include='number').corr()['Survived'].sort_values().round(3))

**What I see:** After cleaning, `df_clean` has 0 missing values (I filled `Age` with the median 28.0 and `Embarked` with `'S'`). The plots and tables show:
- **Sex** has the strongest link: **74.2%** of the women survived but only **18.9%** of the men.
- **Class:** survival goes down from 1st to 3rd class (the correlation of `Pclass` with `Survived` is -0.338). In 1st class 96.8% of the women survived, in 3rd class only 50%.
- **Fare** goes up with survival (correlation 0.257), because the expensive tickets were mostly 1st class.
- **Embarked:** 55.4% of the passengers from Cherbourg (C) survived, against 33.9% from Southampton (S).
- **FamilySize:** passengers traveling alone survived about 30% of the time, families of 2 to 4 people 55% to 72%, and big families (5 or more) much less.
- **Age** alone has a weak straight-line link (correlation -0.065), but the age histogram shows that young children survived more often than adults.

In [ ]:
# I split df_clean into features X and target y ('Survived'), then I do a stratified train-test split with test_size=0.2 and random_state=42.
# I name the outputs X_train, X_test, y_train, y_test and print their shapes in a complete sentence.

from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=['Survived'])  # everything except the answer
y = df_clean['Survived']                 # the answer
# stratify=y keeps the same share of survivors in the training set and in the testing set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'The training set has {X_train.shape[0]} passengers and {X_train.shape[1]} features, '
      f'and the testing set has {X_test.shape[0]} passengers and {X_test.shape[1]} features. '
      f'y_train has {y_train.shape[0]} answers and y_test has {y_test.shape[0]} answers.')

### Learning point
Models read numbers. Use a preprocessing step to convert text columns to numbers and to fill any missing values. Pipelines keep preprocessing attached to the model so you do not leak information.

In [ ]:
# This piece of code is already prefilled, run it to execute it and see the results.
# It defines a helper to build a preprocessing transformer for numeric and categorical columns.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def make_preprocessor(num_cols, cat_cols, scale_numeric=True):
    num_steps = []
    num_steps.append(("imputer", SimpleImputer(strategy="median")))
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    num_pipe = Pipeline(steps=num_steps)

    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    pre = ColumnTransformer(transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])
    return pre

## Exercise 2: Decision Tree Classifier without Grid Search

### Instructions
Implement a Decision Tree Classifier on the Titanic dataset.
Manually choose and set the hyperparameters.
Evaluate its performance using accuracy, precision, recall, and F1 score.

In this exercise, I build a Decision Tree with settings I choose myself, train it, and evaluate it with accuracy, precision, recall and F1.

### What I learn here
- A decision tree asks a chain of yes/no questions, like "Is the passenger female?".
- Trees do not need scaled numbers, so I turn the scaling off for them.
- `max_depth` and `min_samples_leaf` keep the tree small, so it does not memorize the training set.

**My choices and why:** `max_depth=4` lets the tree ask at most 4 questions in a row, enough for patterns like sex, class and age. `min_samples_leaf=5` means every final group (leaf) has at least 5 passengers, so the tree cannot make a rule for one single passenger.

### Learning point
Decision trees work on raw numbers and one-hot encoded categories. Deeper trees can overfit. Start simple and grow only if needed.

In [ ]:
# This piece of code is already prefilled, run it to execute it and see the results.
# It defines a function to print metrics and draw a confusion matrix.

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def report_classification(y_true, y_pred):
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
    print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
    print("F1 score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred)
    plt.title("Confusion Matrix")
    plt.show()

In [ ]:
# I choose the numeric and categorical columns from X_train and build a preprocessing transformer with make_preprocessor.
# I create a Pipeline that includes the preprocessor and a DecisionTreeClassifier with my chosen hyperparameters.
# I fit the pipeline, predict on X_test, and call report_classification.

from sklearn.tree import DecisionTreeClassifier

num_cols = X_train.select_dtypes(include='number').columns.tolist()   # Pclass, Age, SibSp, Parch, Fare, FamilySize
cat_cols = X_train.select_dtypes(exclude='number').columns.tolist()   # Sex, Embarked
print('Numeric:', num_cols)
print('Categorical:', cat_cols)

# scale_numeric=False: a tree only compares a number with a threshold, so scaling changes nothing.
pre_tree = make_preprocessor(num_cols, cat_cols, scale_numeric=False)

dt_pipe = Pipeline([
    ('pre', pre_tree),
    ('dt', DecisionTreeClassifier(max_depth=4, min_samples_leaf=5, random_state=42)),
])
dt_pipe.fit(X_train, y_train)
y_pred_dt = dt_pipe.predict(X_test)
report_classification(y_test, y_pred_dt)

In this cell, I draw which features my tree uses the most (feature importances), as the "What will you create" list asks.

In [ ]:
# get_feature_names_out gives the column names after one-hot encoding, for example 'cat__Sex_female'.
feature_names = dt_pipe.named_steps['pre'].get_feature_names_out()
importances = pd.Series(dt_pipe.named_steps['dt'].feature_importances_, index=feature_names).sort_values()
importances.plot(kind='barh', figsize=(7, 4), title='Decision Tree feature importances')
plt.xlabel('Importance (the shares add up to 1)')
plt.show()

**My result:** My tree gets **accuracy 0.7765**, precision 0.8085, recall 0.5507 and **F1 0.6552**. The precision is high: when it says "survived", it is right 81% of the time. But the recall is low: it finds only 38 of the 69 survivors in the test set. The feature importances show that the tree mainly uses `Sex_female` (0.601), then `Pclass` (0.204) and `Age` (0.089), which matches what I saw in the EDA.

## Exercise 3: Decision Tree Classifier with Grid Search

### Instructions
Apply GridSearchCV to find the optimal hyperparameters for the Decision Tree Classifier.
Compare its performance with the manually tuned model from Exercise 2.

In this exercise, I let GridSearchCV try a small grid of tree settings with 5-fold cross-validation, then I evaluate the best tree on the test set and compare it with my tree from Exercise 2.

### What I learn here
- `best_params_` is the winning combination and `best_score_` is its average validation accuracy, not the test accuracy.
- A better validation score does not always mean a better test score.

### Learning point
Grid search tries a small grid of settings and keeps the best. Limit the grid so you learn fast and avoid long runs.

In [ ]:
# I build a Pipeline with the same preprocessor and a DecisionTreeClassifier.
# I define a small parameter grid over max_depth, min_samples_split, and min_samples_leaf.
# I run GridSearchCV with cv=5, print best_params_ and best_score_, and evaluate best_estimator_ on the test set.

from sklearn.model_selection import GridSearchCV

dt_pipe_cv = Pipeline([
    ('pre', pre_tree),
    ('dt', DecisionTreeClassifier(random_state=42)),
])
dt_grid = {
    'dt__max_depth': [3, 4, 5, 6, 8, None],   # None = no depth limit
    'dt__min_samples_split': [2, 10, 20],
    'dt__min_samples_leaf': [1, 5, 10],
}
# 6 x 3 x 3 = 54 combinations, and with cv=5 that is 270 fits (a few seconds).
dt_search = GridSearchCV(dt_pipe_cv, dt_grid, cv=5, scoring='accuracy', n_jobs=-1)
dt_search.fit(X_train, y_train)
print('Best params:', dt_search.best_params_)
print('Best cross-validation accuracy:', round(dt_search.best_score_, 4))

y_pred_dt_gs = dt_search.best_estimator_.predict(X_test)
report_classification(y_test, y_pred_dt_gs)

# I compare the two trees side by side on the test set.
print('Exercise 2 tree test accuracy:', round(accuracy_score(y_test, y_pred_dt), 4), '| F1:', round(f1_score(y_test, y_pred_dt), 4))
print('Exercise 3 tree test accuracy:', round(accuracy_score(y_test, y_pred_dt_gs), 4), '| F1:', round(f1_score(y_test, y_pred_dt_gs), 4))

**My result:** The best tree has `max_depth=6`, `min_samples_leaf=1` and `min_samples_split=10`, with an average cross-validation accuracy of **0.8217**. On the test set, it is slightly better than my tree from Exercise 2: **accuracy 0.7821** instead of 0.7765 and **F1 0.6723** instead of 0.6552 (it finds 40 survivors instead of 38). The grid allowed a deeper tree, and `min_samples_split=10` still stops it from splitting very small groups. The test accuracy (0.7821) is lower than the cross-validation score (0.8217), which is normal: the test set has only 179 passengers, so one passenger changes the accuracy by about 0.56%.

## Exercise 4: K-Nearest Neighbors (KNN) without Grid Search

### Instructions
Train a KNN classifier on the Titanic dataset without using hyperparameter tuning.
Choose the number of neighbors and distance metric based on your understanding.
Assess the model’s performance and discuss the choice of hyperparameters.

In this exercise, I train a KNN with a number of neighbors and a distance that I choose myself, evaluate it, and explain my choice.

### What I learn here
- KNN predicts a passenger by looking at the k most similar passengers in the training set and taking a vote.
- KNN measures distances, so the number columns must be scaled.

**My choices and why:** `n_neighbors=7`: an odd number, so a vote between the two classes can never be a tie, and big enough that one unusual neighbor does not decide alone. `metric='minkowski'` with the default `p=2` is the normal straight-line (Euclidean) distance, which works well once all columns are scaled.

### Learning point
KNN is simple. It predicts by voting among nearby points. Scale numeric features or distances will be dominated by large-scale columns.

In [ ]:
# I build a Pipeline with make_preprocessor (with numeric scaling turned on) and KNeighborsClassifier.
# I choose n_neighbors and metric, fit on the training set, predict on the test set, and report the metrics.

from sklearn.neighbors import KNeighborsClassifier

# scale_numeric=True: without scaling, Fare (0 to 512) would count far more than Pclass (1 to 3) in the distance.
pre_scaled = make_preprocessor(num_cols, cat_cols, scale_numeric=True)

knn_pipe = Pipeline([
    ('pre', pre_scaled),
    ('knn', KNeighborsClassifier(n_neighbors=7, metric='minkowski', p=2)),
])
knn_pipe.fit(X_train, y_train)
y_pred_knn = knn_pipe.predict(X_test)
report_classification(y_test, y_pred_knn)

**My result:** My KNN with 7 neighbors and the straight-line distance gets **accuracy 0.8101**, precision 0.7692, recall 0.7246 and **F1 0.7463**. It finds 50 of the 69 survivors, many more than the trees (38 and 40). After scaling, every feature counts the same in the distance, and 7 neighbors give a stable vote, so this simple choice works well.

## Exercise 5: K-Nearest Neighbors (KNN) with Grid Search

### Instructions
Use GridSearchCV to optimize the hyperparameters of the KNN classifier, like the number of neighbors and distance metric.
Evaluate and compare the performance of the tuned model against the model from Exercise 4.

In this exercise, I let GridSearchCV try several numbers of neighbors and two distances, then I evaluate the best KNN and compare it with my KNN from Exercise 4.

### What I learn here
- A small k follows single points closely (noisy), and a large k smooths everything (it can miss local patterns).
- `manhattan` distance adds the differences of each column, `minkowski` with p=2 is the straight-line distance.

### Learning point
Try a few k values. Too small can be noisy, too large can wash out local patterns. Cross-validation helps you choose.

In [ ]:
# I build a Pipeline with the preprocessor and KNeighborsClassifier.
# I define a parameter grid for n_neighbors and metric (['minkowski','manhattan']).
# I run GridSearchCV with cv=5, print best_params_ and best_score_, and evaluate best_estimator_ on X_test.

knn_pipe_cv = Pipeline([
    ('pre', pre_scaled),
    ('knn', KNeighborsClassifier()),
])
knn_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'knn__metric': ['minkowski', 'manhattan'],
}
# 7 x 2 = 14 combinations, and with cv=5 that is 70 fits.
knn_search = GridSearchCV(knn_pipe_cv, knn_grid, cv=5, scoring='accuracy', n_jobs=-1)
knn_search.fit(X_train, y_train)
print('Best params:', knn_search.best_params_)
print('Best cross-validation accuracy:', round(knn_search.best_score_, 4))

y_pred_knn_gs = knn_search.best_estimator_.predict(X_test)
report_classification(y_test, y_pred_knn_gs)

print('Exercise 4 KNN test accuracy:', round(accuracy_score(y_test, y_pred_knn), 4), '| F1:', round(f1_score(y_test, y_pred_knn), 4))
print('Exercise 5 KNN test accuracy:', round(accuracy_score(y_test, y_pred_knn_gs), 4), '| F1:', round(f1_score(y_test, y_pred_knn_gs), 4))

**My result:** The best settings are `n_neighbors=9` with the `manhattan` distance, with an average cross-validation accuracy of **0.8245**. But on the test set, the tuned KNN is a little worse than mine: **accuracy 0.8045** instead of 0.8101 and **F1 0.7287** instead of 0.7463 (it finds 47 survivors instead of 50). The grid chooses the settings that are best on average over the 5 validation folds, and those are not always the best on one particular test set. The accuracy difference is only one passenger, so the two models are about equally good.

## Exercise 6: Neural Network Classifier without Hyperparameter Tuning

### Instructions
Build a basic Neural Network to classify Titanic passengers.
Set the layers, neurons, and activation functions based on your intuition.
Analyze the model’s effectiveness in predicting survival.

The course page says: "Build a basic Neural Network using libraries like TensorFlow or PyTorch to classify Titanic passengers." The template asks for scikit-learn's `MLPClassifier`, which is also a neural network, and asks me to write in a comment how I would do it with TensorFlow. So I use `MLPClassifier` and I add the TensorFlow (Keras) version as a comment.

### What I learn here
- A neural network has layers of neurons. Each neuron multiplies its inputs by weights, adds them, and passes the result through an activation function.
- `relu` keeps positive values and changes negative values to 0.

**My choices and why:** two hidden layers with 16 and 8 neurons: the data has only 10 input columns after encoding and about 700 training passengers, so a small network is enough and overfits less. `activation='relu'` is the usual choice for hidden layers. `alpha=0.001` adds a small penalty on big weights (regularization). `max_iter=1000` gives the training enough rounds to finish.

### Learning point
Neural networks need scaled inputs and enough data. Start small. A compact model often learns faster and avoids overfitting.

In [ ]:
# I build a Pipeline with make_preprocessor (scaling on) and MLPClassifier.
# I choose hidden_layer_sizes, activation, alpha, and max_iter. I fit on the training data, predict on the test data, and report the metrics.
# With TensorFlow, I would replace MLPClassifier like this (same layers, same activation):
#   from tensorflow import keras
#   X_train_nn = pre_scaled.fit_transform(X_train); X_test_nn = pre_scaled.transform(X_test)
#   model = keras.Sequential([keras.Input(shape=(X_train_nn.shape[1],)),
#                             keras.layers.Dense(16, activation='relu'),
#                             keras.layers.Dense(8, activation='relu'),
#                             keras.layers.Dense(1, activation='sigmoid')])   # 1 output = probability of survival
#   model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
#   model.fit(X_train_nn, y_train, epochs=100, batch_size=32, verbose=0)
#   y_pred_nn = (model.predict(X_test_nn) > 0.5).astype(int).ravel()

from sklearn.neural_network import MLPClassifier

mlp_pipe = Pipeline([
    ('pre', pre_scaled),
    ('mlp', MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu', alpha=0.001,
                          max_iter=1000, random_state=42)),
])
mlp_pipe.fit(X_train, y_train)
y_pred_mlp = mlp_pipe.predict(X_test)
report_classification(y_test, y_pred_mlp)
# I also check the training accuracy: a much higher training score than test score means overfitting.
print('Training accuracy:', round(mlp_pipe.score(X_train, y_train), 4))

**My result:** My network gets **accuracy 0.7765**, precision 0.7302, recall 0.6667 and **F1 0.697**. Its training accuracy is 0.8638, clearly higher than its test accuracy, so it learned some details of the training passengers that do not repeat in new passengers (a little overfitting). It finds 46 of the 69 survivors, more than the trees but fewer than KNN.

## Exercise 7 (Optional): Neural Network Classifier with Hyperparameter Tuning

### Instructions
Implement a Neural Network and use techniques like RandomizedSearchCV (or other applicable methods) to tune hyperparameters like the number of layers, neurons, and learning rate.
Evaluate the performance and compare it with the Neural Network from Exercise 6.

The course page adds: "(To do this exercise, you must use the GPU from Google Colab)". The template's version uses scikit-learn's `MLPClassifier`, which always runs on the CPU, and this small search finishes in less than a minute, so it also works without a GPU.

In this exercise, I let RandomizedSearchCV try 15 random combinations of layers, neurons, `alpha` and learning rate with 3-fold cross-validation, and I compare the best network with my network from Exercise 6.

### What I learn here
- RandomizedSearchCV does not try every combination. It picks `n_iter` random ones, which is faster when there are many possible combinations.
- `hidden_layer_sizes=(32, 16)` means 2 hidden layers with 32 and 16 neurons, and `(32,)` means 1 hidden layer.

### Learning point
Tune a few knobs at a time. Use randomized search for a quick scan. Keep iterations small so you see results sooner.

In [ ]:
# I wrap my MLP pipeline in RandomizedSearchCV with a small param distribution over hidden_layer_sizes, alpha, and learning_rate_init.
# I use cv=3 and a modest n_iter, print best_params_ and best_score_, and evaluate the best_estimator_ on the test set.
# With TensorFlow in Colab, I would wrap a function that builds the Keras model in scikeras.wrappers.KerasClassifier;
# that wrapper behaves like a scikit-learn model, so I could pass it to RandomizedSearchCV in the same way (this is where the GPU helps).

from sklearn.model_selection import RandomizedSearchCV
import numpy as np

mlp_pipe_cv = Pipeline([
    ('pre', pre_scaled),
    ('mlp', MLPClassifier(activation='relu', max_iter=2000, random_state=42)),
])
mlp_dist = {
    'mlp__hidden_layer_sizes': [(8,), (16,), (32,), (16, 8), (32, 16), (64, 32), (32, 16, 8)],  # number of layers and neurons
    'mlp__alpha': [0.0001, 0.001, 0.01, 0.1, 1.0],                # penalty on big weights
    'mlp__learning_rate_init': [0.0005, 0.001, 0.005, 0.01],      # size of each learning step
}
# 7 x 5 x 4 = 140 possible combinations, but I only try n_iter=15 of them: 15 x 3 folds = 45 fits.
mlp_search = RandomizedSearchCV(mlp_pipe_cv, mlp_dist, n_iter=15, cv=3, scoring='accuracy',
                                random_state=42, n_jobs=-1)
mlp_search.fit(X_train, y_train)
print('Best params:', mlp_search.best_params_)
print('Best cross-validation accuracy:', round(mlp_search.best_score_, 4))

y_pred_mlp_rs = mlp_search.best_estimator_.predict(X_test)
report_classification(y_test, y_pred_mlp_rs)

print('Exercise 6 network test accuracy:', round(accuracy_score(y_test, y_pred_mlp), 4), '| F1:', round(f1_score(y_test, y_pred_mlp), 4))
print('Exercise 7 network test accuracy:', round(accuracy_score(y_test, y_pred_mlp_rs), 4), '| F1:', round(f1_score(y_test, y_pred_mlp_rs), 4))

**My result:** The best network has 1 hidden layer with 16 neurons, `alpha=1.0` and `learning_rate_init=0.01`, with an average cross-validation accuracy of **0.8287**. On the test set, it is better than my network from Exercise 6: **accuracy 0.8101** instead of 0.7765 and **F1 0.7167** instead of 0.697. The search chose a smaller network and the biggest penalty on big weights (`alpha=1.0`), and both reduce the overfitting I saw in Exercise 6.

## Comparison of all my models

In this cell, I put the test metrics of all six models in one table and one bar chart, so I can compare the models with and without tuning and choose the best one.

In [ ]:
# I keep the predictions of every model in a dictionary: name -> predictions on X_test.
all_preds = {
    'Tree (manual)': y_pred_dt,
    'Tree (grid)': y_pred_dt_gs,
    'KNN (manual)': y_pred_knn,
    'KNN (grid)': y_pred_knn_gs,
    'NN (manual)': y_pred_mlp,
    'NN (random search)': y_pred_mlp_rs,
}
rows = {}
for name, pred in all_preds.items():
    rows[name] = {
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
    }
comparison = pd.DataFrame(rows).T.round(4)
display(comparison.sort_values('f1', ascending=False))

comparison.plot(kind='bar', figsize=(11, 4))
plt.title('Test metrics of my six Titanic models')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(loc='lower right')
plt.show()

**My comparison report:**
- **Best model:** my KNN from Exercise 4 has the best F1 (**0.7463**) and the best recall (0.7246), and it ties for the best accuracy (**0.8101**) with the tuned neural network. The tuned network has the best precision (0.8431) but finds fewer survivors.
- **Tuning:** tuning helped the Decision Tree a little (F1 0.6552 → 0.6723) and the neural network the most (F1 0.697 → 0.7167), but not KNN (F1 0.7463 → 0.7287). A higher cross-validation score does not always mean a better test score.
- **Preprocessing per model:** the trees work without scaling, but KNN and the neural network need scaled numbers, so I used `scale_numeric=True` for them.
- **Trees:** both trees have high precision but low recall. They are careful, and they miss many survivors.
- My choice is **KNN with 7 neighbors**: it has the best balance between precision and recall, and it is simple. With only 179 test passengers, the differences between the top models are only a few passengers, so I would check them with cross-validation before choosing for sure.